# HAPO-BSS: Health-Aware Predictive Optimization of EV Battery Swapping Stations
### Under stochastic demand and renewable-energy uncertainty

This notebook is an end-to-end, runnable implementation of the HAPO-BSS framework:

| Module | Role | Equations |
|---|---|---|
| Data acquisition + pre-processing | imputation, IQR outliers, min-max, station state | (1)-(6) |
| **TDPM** | uncertainty-aware temporal demand prediction | (7)-(12) |
| **RGEMM** | renewable forecast + renewable/grid energy balance | (13)-(16) |
| **BHAAM** | SOC/SOH/temperature/cycle health assessment + degradation estimator | (17)-(22) |
| **BIEM** | predictive inventory evaluation (deficit / surplus) | (23)-(27) |
| **PBAM** | predictive, compatibility- and reserve-aware allocation (Hungarian) | (28)-(33) |
| Charging & coordination | charger-capacity-limited, renewable/price-aware | (34)-(36) |
| **AMOO** | adaptive multi-objective optimisation + feedback | (37)-(44), (52)-(53) |
| Constraints | feasibility checks | (45)-(51) |

**Important notes on what is real and what is simulated**
* The EVIoT Kaggle dataset (`datasetengineer/eviot-predictivemaintdataset`) contains battery telemetry only. It is used (if found) to initialise the battery fleet (SOC/SOH/temperature/cycles). Swap demand, PV generation, tariffs and weather are **simulated** by a documented stochastic generator, as stated in the paper's experimental setup ("supplemented with clearly documented BSS demand, energy and inventory data").
* All numbers produced here are computed by the code. They will **not** be identical to the figures quoted in the paper's abstract; they depend on the simulator parameters.
* The comparison methods GSO-PPO, LSTM-RHO, AGWO and MILP are **lightweight surrogates** (see section 9), not reproductions of the original papers' code.

In [ ]:
import os, time, math, glob, warnings
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment, linprog
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
warnings.filterwarnings('ignore')

try:
    import torch, torch.nn as nn
    USE_TORCH = True
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
except Exception:                      # CPU-only fallback: sklearn models replace the deep nets
    USE_TORCH, DEVICE = False, 'cpu'
print('PyTorch available:', USE_TORCH, '| device:', DEVICE)

SEED = 42
np.random.seed(SEED)
if USE_TORCH: torch.manual_seed(SEED)
os.makedirs('figures', exist_ok=True)

def show(name):
    plt.tight_layout(); plt.savefig(f'figures/{name}.png', dpi=130); plt.show()

CFG = dict(
    spd=24, train_days=105, val_days=14, test_days=21,        # forecasting data split
    L=24, H=4,                                                # look-back window / prediction horizon (hours)
    n_batt=60, cap_kwh=60.0, p_max=520.0, c_max=1.0, eta=0.95, aux_kw=20.0, g_max=700.0,
    soc_target=0.95, soc_req=0.90, soh_min=0.70, temp_max=55.0,
    pv_kwp=700.0, abandon_steps=2,
    k_cyc=2.0e-4, k_use=1.5e-4, pack_value=7200.0, lost_penalty=60.0,   # degradation / cost model
    sim_days=14, n_scen=3,                                    # station-simulation episodes
)
ETYPE_P = np.array([0.5, 0.3, 0.2])                           # EV (pack) type mix
COMPAT = np.array([[1,0,0],[0,1,1],[0,0,1]], dtype=bool)      # COMPAT[ev_type, battery_type]  (compatibility matrix)

## 1. Data acquisition
Four groups of data (Section 4.2): EV demand, battery operational data, energy/electricity data, environmental/temporal data.

In [ ]:
def gauss(x, m, s): return np.exp(-0.5*((x-m)/s)**2)

def make_series(days, seed, spd=24):
    """Stochastic hourly BSS environment: swap demand, PV, tariff, weather (documented simulator)."""
    r = np.random.default_rng(seed)
    T = days*spd; t = np.arange(T); hour = t % spd; day = t // spd; dow = day % 7
    cloud = np.zeros(T); c = 0.35
    for i in range(T):
        c = 0.92*c + 0.08*0.35 + 0.10*r.normal(); cloud[i] = c
    cloud = np.clip(cloud, 0, 1)
    tn = 0.0; temp = np.zeros(T)
    for i in range(T):
        tn = 0.9*tn + 0.5*r.normal(); temp[i] = 27 + 5*np.sin(2*np.pi*(hour[i]-9)/24) + 2*np.sin(2*np.pi*day[i]/40) + tn
    clear = np.maximum(0, np.sin(np.pi*(hour-6)/12))**1.2
    pv = np.clip(CFG['pv_kwp']*clear*(1-0.8*cloud) + r.normal(0, 4, T)*(clear > 0), 0, None)
    event = 1 + 0.25*(r.random(days) < 0.08)                         # random surge days
    dayf = np.exp(r.normal(0, 0.08, days))
    wk = np.where(dow >= 5, 0.85, 1.0)
    base = 3 + 7*gauss(hour, 8.5, 1.8) + 9*gauss(hour, 18, 2.2) + 3*gauss(hour, 13, 3)
    lam = base*wk*(1 + 0.12*(cloud > 0.75))*(1 + 0.01*(temp-27))*event[day]*dayf[day]
    demand = r.poisson(lam).astype(float)
    tou = np.select([hour < 6, hour < 10, hour < 16, hour < 22], [0.09, 0.14, 0.12, 0.24], 0.11)
    pn = 0.0; price = np.zeros(T)
    for i in range(T):
        pn = 0.8*pn + 0.02*r.normal(); price[i] = tou[i]*(1 + pn)
    return pd.DataFrame(dict(demand=demand, pv=pv, price=price, temp=temp, cloud=cloud, hour=hour, dow=dow))

def pick(cols, keys):
    low = {c: c.lower().replace(' ', '_') for c in cols}
    for k in keys:
        for c, l in low.items():
            if k in l: return c
    return None

EV_COLS = dict(soc=['state_of_charge', 'soc'], soh=['state_of_health', 'soh'],
               temp=['battery_temp', 'temperature'], cyc=['charging_cycle', 'cycle'])

def load_eviot():
    paths = []
    for pat in ['/kaggle/input/**/*.csv', './**/*eviot*.csv', './data/**/*.csv', '/content/**/*eviot*.csv']:
        paths += glob.glob(pat, recursive=True)
    paths = [p for p in paths if 'eviot' in p.lower() or 'predictive' in p.lower()]
    if not paths:
        try:
            import kagglehub
            d = kagglehub.dataset_download('datasetengineer/eviot-predictivemaintdataset')
            paths = glob.glob(os.path.join(d, '**', '*.csv'), recursive=True)
        except Exception as e:
            print('EVIoT dataset not found / not downloadable (%s) -> synthetic battery fleet.' % type(e).__name__)
            return None
    try:
        df = pd.read_csv(paths[0]); print('EVIoT loaded:', paths[0], df.shape)
        print('mapped columns:', {k: pick(df.columns, v) for k, v in EV_COLS.items()})
        return df
    except Exception as e:
        print('Could not read EVIoT csv:', e); return None

EVIOT = load_eviot()

def init_fleet(rng, evdf=None, n=None):
    """Initial battery fleet; columns taken from EVIoT when available, otherwise synthetic."""
    n = n or CFG['n_batt']
    soc = rng.uniform(0.45, 0.96, n)
    soh = np.clip(rng.beta(4, 3, n)*0.28 + 0.74, 0.74, 1.0)
    temp = rng.normal(30, 2, n); cyc = rng.uniform(50, 600, n)
    if evdf is not None:
        samp = evdf.sample(n, replace=True, random_state=int(rng.integers(1_000_000))).reset_index(drop=True)
        def col(key):
            c = pick(samp.columns, EV_COLS[key])
            if c is None or not pd.api.types.is_numeric_dtype(samp[c]): return None
            v = samp[c].astype(float).values
            return v if np.nanstd(v) > 1e-9 else None
        v = col('soh');  soh = np.clip(v/100 if v.max() > 1.5 else v, 0.74, 1.0) if v is not None else soh
        v = col('temp'); temp = np.clip(v, 20, 45) if v is not None else temp
        v = col('cyc');  cyc = v - v.min() if v is not None else cyc
        v = col('soc');  soc = np.clip(v/100 if v.max() > 1.5 else v, 0.45, 0.96) if v is not None else soc
    typ = rng.choice(3, size=n, p=ETYPE_P)
    return dict(soc=soc.astype(float), soh=soh.astype(float), temp=temp.astype(float), cyc=cyc.astype(float), typ=typ)

total_days = CFG['train_days'] + CFG['val_days'] + CFG['test_days']
raw = make_series(total_days, seed=7)
print(raw.describe().round(2).T[['mean', 'std', 'min', 'max']])

## 2. Data pre-processing — equations (1)–(6)
Sensor faults (missing values, spikes) are injected into the raw series to show the pipeline working, then removed by
(1) IQR outlier detection -> (2) temporal linear interpolation -> (3) min-max normalisation (fitted on the training part only).

In [ ]:
def inject_faults(df, r, miss=0.015, out=0.005, cols=('demand', 'pv', 'price', 'temp')):
    d = df.copy(); n = len(d)
    for c in cols:
        v = d[c].values.copy()
        o = np.where(r.random(n) < out)[0]
        v[o] = np.nan_to_num(v[o])*r.uniform(2.5, 4, len(o)) + r.uniform(10, 40, len(o))
        v[r.random(n) < miss] = np.nan
        d[c] = v
    return d

def linear_impute(x):
    """Eq. (1): x_t = x_prev + (x_next - x_prev) * (t - t_prev)/(t_next - t_prev)."""
    x = np.asarray(x, float).copy(); idx = np.arange(len(x)); good = ~np.isnan(x)
    return np.interp(idx, idx[good], x[good])

def iqr_bounds(x):
    """Eqs. (2)-(4): IQR = Q3-Q1, lower = Q1-1.5 IQR, upper = Q3+1.5 IQR."""
    q1, q3 = np.nanpercentile(x, [25, 75]); iqr = q3 - q1
    return q1 - 1.5*iqr, q3 + 1.5*iqr

def clean(df, cols=('demand', 'pv', 'price', 'temp')):
    d = df.copy(); stats = {}
    for c in cols:
        v = d[c].values.astype(float); n_nan = int(np.isnan(v).sum())
        lo, hi = iqr_bounds(v); out = (v < lo) | (v > hi); v[out] = np.nan
        d[c] = linear_impute(v); stats[c] = (n_nan, int(out.sum()))
    return d, stats

faulty = inject_faults(raw, np.random.default_rng(1))
cleaned, st = clean(faulty)
print('(missing, outliers) per variable:', st)
for c in ['demand', 'pv', 'price', 'temp']:
    print(f'{c:7s} MAE vs ground truth  faulty(NaN->0): {np.abs(np.nan_to_num(faulty[c])-raw[c]).mean():8.3f}   cleaned: {np.abs(cleaned[c]-raw[c]).mean():8.3f}')

n_tr = CFG['train_days']*CFG['spd']; n_va = n_tr + CFG['val_days']*CFG['spd']
def fit_minmax(df, cols, n_fit):
    return {c: (float(df[c].iloc[:n_fit].min()), float(df[c].iloc[:n_fit].max())) for c in cols}
MM = fit_minmax(cleaned.assign(util=cleaned.demand.rolling(6, min_periods=1).mean()),
                ['demand', 'pv', 'price', 'temp', 'cloud', 'util'], n_tr)

FEATS = ['demand', 'pv', 'price', 'temp', 'cloud', 'util', 'hs', 'hc', 'ds', 'dc']
def featurize(df):
    """Eq. (5) min-max scaling + cyclic calendar encoding; returns normalised frame."""
    z = df.copy(); z['util'] = z.demand.rolling(6, min_periods=1).mean()
    for c, (lo, hi) in MM.items(): z[c] = (z[c] - lo)/(hi - lo + 1e-9)
    z['hs'] = np.sin(2*np.pi*z.hour/24); z['hc'] = np.cos(2*np.pi*z.hour/24)
    z['ds'] = np.sin(2*np.pi*z.dow/7);   z['dc'] = np.cos(2*np.pi*z.dow/7)
    return z

def make_windows(fz, L, H):
    """History window (L steps) + known future calendar/tariff vector for the H forecast steps."""
    A = fz[FEATS].values.astype(np.float32)
    cal = fz[['hs', 'hc', 'ds', 'dc', 'price']].values.astype(np.float32)
    idx = np.arange(L, len(fz) - H + 1)
    X = np.stack([A[i-L:i] for i in idx]); Fu = np.stack([cal[i:i+H].reshape(-1) for i in idx])
    return X, Fu, idx

FZ = featurize(cleaned)
Xall, Fall, IDX = make_windows(FZ, CFG['L'], CFG['H'])
print('windows:', Xall.shape, Fall.shape)

## 3. TDPM — Temporal Demand Prediction Module, equations (7)–(12)
Eq. (9): $\hat D_{t:t+H-1}=f(D_{t-L:t-1},X_t;\theta)$. Eq. (10): the model also outputs an uncertainty $\hat\sigma$
(heteroscedastic Gaussian head trained with NLL). Eq. (11): $D^{safe}=\hat D+\kappa\hat\sigma$. Eq. (12): $N_{req}=\lceil\sum D^{safe}\rceil$.

TDPM = 2-layer GRU encoder + temporal attention + known-future calendar/tariff vector + (mean, log-variance) head.
Baselines for Table 3: LSTM, CNN, GRU, CNN-LSTM (with PyTorch). Without PyTorch, sklearn models are substituted automatically.

**Note on MAPE:** hourly swap counts are Poisson-like (mean ≈ 7), so hourly MAPE has a high irreducible floor (~30-40 %) for *any* model. WAPE and the MAPE of daily swap volume are reported as well; they are the fairer accuracy measures for count data.

In [ ]:
def mape(y, p, thr=1.0):
    m = y >= thr
    return float(100*np.mean(np.abs((y[m]-p[m])/y[m]))) if m.any() else np.nan
def rmse(y, p): return float(np.sqrt(np.mean((y-p)**2)))

if USE_TORCH:
    class SeqNet(nn.Module):
        def __init__(self, arch, F, H, nfut, hid=64):
            super().__init__(); self.arch = arch; self.H = H
            if arch == 'tdpm':
                self.rnn = nn.GRU(F, hid, num_layers=2, batch_first=True, dropout=0.1); self.attn = nn.Linear(hid, 1)
            elif arch == 'gru':  self.rnn = nn.GRU(F, hid, batch_first=True)
            elif arch == 'lstm': self.rnn = nn.LSTM(F, hid, batch_first=True)
            elif arch == 'cnn':
                self.conv = nn.Sequential(nn.Conv1d(F, hid, 3, padding=1), nn.ReLU(), nn.Conv1d(hid, hid, 3, padding=1), nn.ReLU())
            elif arch == 'cnnlstm':
                self.conv = nn.Sequential(nn.Conv1d(F, hid, 3, padding=1), nn.ReLU()); self.rnn = nn.LSTM(hid, hid, batch_first=True)
            self.head = nn.Sequential(nn.Linear(hid + nfut, hid), nn.ReLU(), nn.Linear(hid, 2*H))
        def forward(self, x, fu):
            a = self.arch
            if a == 'cnn': h = self.conv(x.transpose(1, 2)).mean(-1)
            elif a == 'cnnlstm':
                z = self.conv(x.transpose(1, 2)).transpose(1, 2); o, _ = self.rnn(z); h = o[:, -1]
            else:
                o, _ = self.rnn(x)
                if a == 'tdpm':
                    w = torch.softmax(self.attn(o), dim=1); h = (w*o).sum(1)
                else: h = o[:, -1]
            out = self.head(torch.cat([h, fu], 1))
            return out[:, :self.H], out[:, self.H:].clamp(-6, 2)

class Forecaster:
    """Unified forecaster: returns (mean, sigma) in real units for an H-step horizon."""
    def __init__(self, arch, H, ymin, ymax, thr=1.0, epochs=60, patience=10, seed=0):
        self.arch, self.H, self.ymin, self.ymax, self.thr = arch, H, ymin, ymax, thr
        self.epochs, self.patience, self.seed = epochs, patience, seed
        self.history = dict(loss=[], val_mape=[]); self.sig_scale = 1.0; self.fit_time = 0.0
    def _n(self, y): return (y - self.ymin)/(self.ymax - self.ymin + 1e-9)
    def _r(self, y): return y*(self.ymax - self.ymin + 1e-9) + self.ymin
    def _is_torch(self): return USE_TORCH and self.arch in ('tdpm', 'gru', 'lstm', 'cnn', 'cnnlstm')

    def fit(self, Xtr, Ftr, Ytr, Xva, Fva, Yva):
        t0 = time.time(); Ytr_n, Yva_n = self._n(Ytr), self._n(Yva)
        if self._is_torch(): self._fit_torch(Xtr, Ftr, Ytr_n, Xva, Fva, Yva_n, Yva)
        else:
            flat = lambda X, F: np.concatenate([X.reshape(len(X), -1), F], 1)
            if self.arch == 'ridge': self.m = Ridge(alpha=1.0)
            elif self.arch == 'rf':  self.m = RandomForestRegressor(n_estimators=100, max_depth=14, n_jobs=-1, random_state=self.seed)
            else: self.m = MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=300, early_stopping=True, random_state=self.seed)
            self.m.fit(flat(Xtr, Ftr), Ytr_n); self._flat = flat
        mu, sg = self._raw(Xva, Fva); res = Yva_n - mu
        if sg is not None: self.sig_scale = float(np.clip(np.sqrt(np.mean((res/(sg+1e-6))**2)), 0.3, 3.0))  # calibrate NLL head
        else: self.res_std = res.std(0)
        self.fit_time = time.time() - t0; return self

    def _fit_torch(self, Xtr, Ftr, Ytr_n, Xva, Fva, Yva_n, Yva):
        torch.manual_seed(self.seed)
        net = SeqNet(self.arch, Xtr.shape[2], self.H, Ftr.shape[1]).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=2e-3, weight_decay=1e-5)
        tt = lambda a: torch.tensor(a, dtype=torch.float32, device=DEVICE)
        Xt, Ft, Yt, Xv, Fv = tt(Xtr), tt(Ftr), tt(Ytr_n), tt(Xva), tt(Fva)
        best, bad, n = (1e9, None), 0, len(Xt)
        for ep in range(self.epochs):
            net.train(); perm = torch.randperm(n, device=DEVICE); tot = 0.0
            for b in range(0, n, 128):
                i = perm[b:b+128]; mu, lv = net(Xt[i], Ft[i]); mse = ((mu - Yt[i])**2).mean()
                loss = mse + 0.1*(0.5*(lv + (Yt[i] - mu)**2/torch.exp(lv))).mean() if self.arch == 'tdpm' else mse
                opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
                tot += loss.item()*len(i)
            net.eval()
            with torch.no_grad(): mv, _ = net(Xv, Fv)
            mv = mv.cpu().numpy(); vm = float(((mv - Yva_n)**2).mean())
            self.history['loss'].append(tot/n); self.history['val_mape'].append(mape(Yva, self._r(mv), self.thr))
            if vm < best[0] - 1e-6: best = (vm, {k: v.clone() for k, v in net.state_dict().items()}); bad = 0
            else:
                bad += 1
                if bad >= self.patience: break
        net.load_state_dict(best[1]); self.net = net.eval()

    def _raw(self, X, Fu):
        if self._is_torch():
            mus, sgs = [], []
            with torch.no_grad():
                for b in range(0, len(X), 512):
                    mu, lv = self.net(torch.tensor(X[b:b+512], device=DEVICE), torch.tensor(Fu[b:b+512], device=DEVICE))
                    mus.append(mu.cpu().numpy()); sgs.append(torch.exp(0.5*lv).cpu().numpy())
            mu = np.concatenate(mus); sg = np.concatenate(sgs)
            return (mu, sg) if self.arch == 'tdpm' else (mu, None)
        return self.m.predict(self._flat(X, Fu)).reshape(len(X), self.H), None

    def predict(self, X, Fu):
        mu, sg = self._raw(X, Fu); span = self.ymax - self.ymin + 1e-9
        sigma = sg*self.sig_scale*span if sg is not None else np.tile(self.res_std*span, (len(mu), 1))
        return np.maximum(self._r(mu), 0), sigma

In [ ]:
L, H = CFG['L'], CFG['H']
tr = IDX <= n_tr - H; va = (IDX >= n_tr) & (IDX <= n_va - H); te = IDX >= n_va
Yd = np.stack([cleaned.demand.values[i:i+H] for i in IDX]); Yp = np.stack([cleaned.pv.values[i:i+H] for i in IDX])
dmin, dmax = MM['demand']; pmin, pmax = MM['pv']

ARCHS = ['tdpm', 'lstm', 'cnn', 'gru', 'cnnlstm'] if USE_TORCH else ['tdpm', 'ridge', 'rf']
EPOCHS = 60 if USE_TORCH else 300
demand_models = {}
for a in ARCHS:
    demand_models[a] = Forecaster(a, H, dmin, dmax, thr=1.0, epochs=EPOCHS, seed=SEED).fit(
        Xall[tr], Fall[tr], Yd[tr], Xall[va], Fall[va], Yd[va])
    print(f'trained demand model [{a}] in {demand_models[a].fit_time:.1f}s')

rows = []
for a, m in demand_models.items():
    mu, sg = m.predict(Xall[te], Fall[te])
    n24 = len(mu)//24*24; ya = Yd[te][:n24, 0].reshape(-1, 24).sum(1); pa = mu[:n24, 0].reshape(-1, 24).sum(1)   # daily swap volume
    rows.append(dict(Model=('TDPM (proposed)' if a == 'tdpm' else a.upper()), MAPE_pct=mape(Yd[te], mu),
                     MAPE_h1_pct=mape(Yd[te][:, 0], mu[:, 0]), WAPE_pct=100*np.abs(Yd[te]-mu).sum()/Yd[te].sum(),
                     MAPE_daily_pct=mape(ya, pa), RMSE_swaps=rmse(Yd[te], mu),
                     train_s=m.fit_time, coverage_1sigma=float(np.mean(np.abs(Yd[te]-mu) <= sg))*100 if a == 'tdpm' else np.nan))
TABLE_FORECAST = pd.DataFrame(rows).round(2); print(TABLE_FORECAST.to_string(index=False))

## 4. RGEMM — Renewable-Grid Energy Management Module, equations (13)–(16)

In [ ]:
pv_model = Forecaster('tdpm', H, pmin, pmax, thr=20.0, epochs=EPOCHS if USE_TORCH else 300, seed=SEED).fit(
    Xall[tr], Fall[tr], Yp[tr], Xall[va], Fall[va], Yp[va])                       # Eq. (13)
mu_pv, _ = pv_model.predict(Xall[te], Fall[te])
print(f'PV forecast: MAPE(>20kW)={mape(Yp[te], mu_pv, 20):.2f}%  RMSE={rmse(Yp[te], mu_pv):.2f} kW')

def renewable_used(R, E): return np.minimum(R, E)                    # Eq. (14)
def grid_residual(E, Ru): return np.maximum(E - Ru, 0.0)             # Eq. (15)
def grid_cost(price, G): return float(np.sum(np.asarray(price)*np.asarray(G)))   # Eq. (16)

### Forecast quality (test segment of the main series)

In [ ]:
mu_d, sg_d = demand_models['tdpm'].predict(Xall[te], Fall[te]); tt_idx = IDX[te]
day0 = slice(0, 24*5)
fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
ax[0].plot(Yd[te][day0, 0], 'k.-', label='actual demand'); ax[0].plot(mu_d[day0, 0], 'r-', label='TDPM (1-h ahead)')
ax[0].fill_between(range(len(mu_d[day0])), (mu_d - sg_d)[day0, 0], (mu_d + sg_d)[day0, 0], color='r', alpha=.2, label='±1σ')
ax[0].set_title('TDPM demand forecast (5 days)'); ax[0].set_xlabel('hour'); ax[0].legend()
ax[1].plot(Yp[te][day0, 0], 'k.-', label='actual PV'); ax[1].plot(mu_pv[day0, 0], 'g-', label='RGEMM forecast')
ax[1].set_title('Renewable (PV) forecast'); ax[1].set_xlabel('hour'); ax[1].legend(); show('fig_forecasts')

if USE_TORCH:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4)); h = demand_models['tdpm'].history
    ax[0].plot(h['val_mape'], 'b'); ax[0].set_title('TDPM validation MAPE (%) vs epoch'); ax[0].set_xlabel('epoch')
    ax[1].plot(h['loss'], 'r'); ax[1].set_title('TDPM training loss vs epoch'); ax[1].set_xlabel('epoch'); show('fig_training')

In [ ]:
def forecast_series(df):
    """Rolling forecasts for every decision time t of a (clean) series: arrays of shape (T, H)."""
    fz = featurize(df); X, Fu, idx = make_windows(fz, L, H); T = len(df)
    out = {k: np.full((T, H), np.nan) for k in ['mu_d', 'sg_d', 'mu_r', 'sg_r', 'mu_b']}
    mu, sg = demand_models['tdpm'].predict(X, Fu); out['mu_d'][idx], out['sg_d'][idx] = mu, sg
    mu, sg = pv_model.predict(X, Fu);              out['mu_r'][idx], out['sg_r'][idx] = mu, sg
    base = 'lstm' if 'lstm' in demand_models else 'ridge'                 # forecaster used by the LSTM-RHO surrogate
    out['mu_b'][idx], _ = demand_models[base].predict(X, Fu)
    return out

## 5. Battery-swapping station simulator
Hourly decisions, 15-minute charging sub-steps (CC-CV taper above 80 % SOC). Each swap: the allocated battery leaves with the EV and a
depleted battery of the same slot returns (SOC 10-25 %), aged by usage. Charging ages batteries with a stress model that depends on
C-rate, temperature, SOH (aging accelerates for weak packs) and high SOC. Unserved EVs queue and abandon after `abandon_steps` hours.
The station state of Eq. (6) is returned by `station_state`.

In [ ]:
class Station:
    def __init__(self, cfg, fleet, df, rng, calib=False):
        self.cfg, self.df, self.rng, self.calib = cfg, df, rng, calib
        self.soc, self.soh, self.temp = fleet['soc'].copy(), fleet['soh'].copy(), fleet['temp'].copy()
        self.cyc, self.typ = fleet['cyc'].copy(), fleet['typ'].copy()
        self.soh0 = float(self.soh.mean()); self.queue = []; self.rec = []; self.log = []
        self.loss_use = 0.0; self.loss_chg = 0.0; self.viol = defaultdict(int)

    def ready_mask(self):
        c = self.cfg; return (self.soc >= c['soc_req']) & (self.soh >= c['soh_min'])

    def station_state(self, t):
        """Eq. (6): s_t = [D, R, price, SOC, SOH, inventory, waiting]."""
        d = self.df
        return dict(D=d.demand.iat[t], R=d.pv.iat[t], price=d.price.iat[t], SOC=self.soc.mean(), SOH=self.soh.mean(),
                    inventory=int(self.ready_mask().sum()), waiting=len(self.queue))

    def _swap(self, b, amb):
        c = self.cfg
        loss = c['k_use']*0.7*(1 + 6*(1 - self.soh[b])**2)*(1 + 0.03*max(self.temp[b] - 30, 0))
        self.soh[b] = max(self.soh[b] - loss, 0.5); self.loss_use += loss; self.cyc[b] += 0.7
        self.soc[b] = self.rng.uniform(0.10, 0.25); self.temp[b] = amb + 4

    def _charge(self, f, c_rate, order, amb):
        """Charging over one hour in 4 sub-steps; power split by priority within budget f*P_max and per-battery cap."""
        cfg = self.cfg; cap = cfg['cap_kwh']; dt = 0.25; E_in = 0.0
        budget = f*cfg['p_max']
        for _ in range(4):
            p = np.zeros_like(self.soc)
            idx = np.where((self.soc < cfg['soc_target'] - 1e-6) & (self.temp < cfg['temp_max']))[0]
            if len(idx):
                key = -self.soc[idx] if order == 'soc' else -(self.soc[idx] - 0.02*np.maximum(self.temp[idx] - 35, 0))
                idx = idx[np.argsort(key)]
                taper = np.where(self.soc[idx] < 0.8, 1.0, np.maximum(0.15, (0.99 - self.soc[idx])/0.19))
                pcap = c_rate*cfg['c_max']*cap*taper
                pcap = np.minimum(pcap, (cfg['soc_target'] - self.soc[idx])*cap*self.soh[idx]/(cfg['eta']*dt))
                cum = np.cumsum(pcap); pi = np.clip(budget - (cum - pcap), 0, pcap); p[idx] = pi
            dsoc = p*cfg['eta']*dt/(cap*self.soh)
            crate = p/cap
            stress = (1 + 1.2*crate**1.5)*(1 + 0.04*np.maximum(self.temp - 30, 0))*(1 + 6*(1 - self.soh)**2)*(1 + 0.8*np.maximum(self.soc - 0.85, 0))
            loss = cfg['k_cyc']*dsoc*stress
            if self.calib:
                for i in np.where(dsoc > 0)[0]:
                    self.log.append((self.soh[i], self.temp[i], crate[i], self.soc[i], loss[i]/dsoc[i]))
            self.soc = np.clip(self.soc + dsoc, 0, 1); self.soh = self.soh - loss; self.loss_chg += loss.sum()
            teq = amb + 18*crate**2; self.temp = self.temp + 0.4*(teq - self.temp)
            if p.sum() > cfg['p_max'] + 1e-6: self.viol['charger_capacity'] += 1
            if self.soc.min() < 0 or self.soc.max() > 1: self.viol['soc_range'] += 1
            E_in += p.sum()*dt
        return E_in

    def step(self, t, policy, ctx):
        cfg, df = self.cfg, self.df
        n_new = int(df.demand.iat[t]); amb = float(df.temp.iat[t]); pv = float(df.pv.iat[t]); price = float(df.price.iat[t])
        evs = self.queue + [(int(self.rng.choice(3, p=ETYPE_P)), t) for _ in range(n_new)]
        self.queue = []; rm = self.ready_mask(); n_ready = int(rm.sum())
        t0 = time.perf_counter(); assign = policy.allocate(self, evs, t, ctx) if evs else []; t_dec = time.perf_counter() - t0
        used = set(); s_imm = s_late = lost = first_fail = 0; wait_min = 0.0
        for (et, ta), b in zip(evs, assign):
            w = t - ta
            ok = (b >= 0) and (b not in used) and COMPAT[et, self.typ[b]] and rm[b]
            if b >= 0 and b in used: self.viol['double_allocation'] += 1
            if ok:
                used.add(b); self._swap(b, amb); wait_min += 60*w
                if w == 0: s_imm += 1
                else: s_late += 1
            else:
                if w == 0: first_fail += 1
                if w >= cfg['abandon_steps']: lost += 1; wait_min += 60*w
                else: self.queue.append((et, ta))
        t0 = time.perf_counter(); f, c, order = policy.charge_action(self, t, ctx); t_dec += time.perf_counter() - t0
        E_in = self._charge(f, c, order, amb)
        E_tot = E_in + cfg['aux_kw']*1.0
        Ru = float(renewable_used(pv, E_tot)); G = float(grid_residual(E_tot, Ru))
        if Ru > pv + 1e-9: self.viol['renewable'] += 1
        if G > cfg['g_max']: self.viol['grid_power'] += 1
        obs = dict(t=t, n_new=n_new, n_evs=len(evs), s_imm=s_imm, s_late=s_late, lost=lost, first_fail=first_fail, wait_min=wait_min,
                   E_in=E_in, E_tot=E_tot, R_used=Ru, grid=G, cost=price*G, pv=pv, price=price, f=f, c=c,
                   inv_ok=int(n_ready >= len(evs)), n_ready=n_ready, soh_mean=float(self.soh.mean()))
        t0 = time.perf_counter(); policy.after_step(self, t, ctx, obs); obs['t_dec'] = t_dec + time.perf_counter() - t0
        self.rec.append(obs); return obs

    def summary(self):
        cfg = self.cfg; r = pd.DataFrame(self.rec); arr = max(r.n_new.sum(), 1); served = r.s_imm.sum() + r.s_late.sum()
        loss = self.loss_use + self.loss_chg
        out = dict(grid_cost=r.cost.sum(), deg_loss_pct=100*loss/len(self.soh), deg_cost=loss*cfg['pack_value'],
                   lost_evs=int(r.lost.sum()), avg_wait_min=r.wait_min.sum()/max(served + r.lost.sum(), 1),
                   BIAR=100*r.inv_ok.mean(), BSR=100*r.first_fail.sum()/arr, SFR=100*r.s_imm.sum()/arr, BASR=100*served/arr,
                   RESSR=100*r.R_used.sum()/r.E_tot.sum(), GEDR=100*r.grid.sum()/r.E_tot.sum(),
                   BHRR=100*self.soh.mean()/self.soh0, CCU=100*r.E_in.sum()/(cfg['p_max']*len(r)), OCT_ms=1000*r.t_dec.mean(),
                   soh_std_pct=100*self.soh.std())
        out['total_cost'] = out['grid_cost'] + out['deg_cost'] + cfg['lost_penalty']*out['lost_evs']
        return out

## 6. BHAAM — Battery Health-Aware Assessment, equations (17)–(22)
Eq. (18): the degradation-impact function $f_d(\cdot;\theta)$ is **learned from charging telemetry** (polynomial ridge regression on a calibration run
with random actions), instead of being hard-coded.

In [ ]:
class DegEstimator:
    def fit(self, log):
        a = np.array(log); X = a[:, :4]; y = np.log(np.clip(a[:, 4], 1e-9, None))
        self.m = make_pipeline(PolynomialFeatures(2), Ridge(alpha=1e-3)).fit(X, y); return self
    def rate(self, soh, temp, crate, soc=0.6):
        soh, temp, crate, soc = np.broadcast_arrays(np.asarray(soh, float), np.asarray(temp, float), np.asarray(crate, float), np.asarray(soc, float))
        return np.exp(self.m.predict(np.stack([soh.ravel(), temp.ravel(), crate.ravel(), soc.ravel()], 1))).reshape(soh.shape)

class RandomPolicy:
    def __init__(self, seed=0): self.r = np.random.default_rng(seed)
    def reset(self, sim, ctx): pass
    def allocate(self, sim, evs, t, ctx): return greedy_soc_alloc(sim, evs)
    def charge_action(self, sim, t, ctx): return self.r.choice([0.3, 0.6, 1.0]), self.r.choice([0.4, 0.7, 1.0]), 'soc'
    def after_step(self, sim, t, ctx, obs): pass

def greedy_soc_alloc(sim, evs):
    """SOC-only allocation: highest-SOC compatible ready battery (the conventional baseline)."""
    avail = sim.ready_mask().copy(); out = []
    for et, _ in evs:
        cand = np.where(avail & COMPAT[et, sim.typ])[0]
        if len(cand): b = cand[np.argmax(sim.soc[cand])]; avail[b] = False
        else: b = -1
        out.append(int(b))
    return out

def minmax01(x):
    x = np.asarray(x, float); r = x.max() - x.min()
    return (x - x.min())/r if r > 1e-12 else np.zeros_like(x)

class BHAAM:
    def __init__(self, deg, w=(0.35, 0.25, 0.25, 0.15)):
        assert abs(sum(w) - 1) < 1e-9 and min(w) >= 0           # Eq. (21)
        self.deg, self.w = deg, w
    def health_state(self, sim):                                 # Eq. (17): h_i = [SOC, SOH, T, cycles]
        return np.stack([sim.soc, sim.soh, sim.temp, sim.cyc], 1)
    def degradation_risk(self, sim):                             # Eq. (18)
        return self.deg.rate(sim.soh, sim.temp, 0.7, 0.6)
    def feasible(self, sim):                                     # Eq. (19) (+ temperature safety)
        c = sim.cfg; return (sim.soc >= c['soc_req']) & (sim.soh >= c['soh_min']) & (sim.temp <= c['temp_max'])
    def terms(self, sim):
        c = sim.cfg
        soc_n = np.clip((sim.soc - c['soc_req'])/(1 - c['soc_req']), 0, 1)
        soh_n = np.clip((sim.soh - c['soh_min'])/(1 - c['soh_min']), 0, 1)
        return soc_n, soh_n, minmax01(self.degradation_risk(sim)), minmax01(sim.cyc)
    def score(self, sim):                                        # Eq. (20)
        soc_n, soh_n, d_n, cy_n = self.terms(sim); w = self.w
        return w[0]*soc_n + w[1]*soh_n - w[2]*d_n - w[3]*cy_n
    def select(self, sim):                                       # Eq. (22) (feasible set only)
        feas = self.feasible(sim)
        if not feas.any(): return -1
        s = np.where(feas, self.score(sim), -np.inf); return int(np.argmax(s))

## 7. BIEM (equations 23–27) and PBAM (equations 28–33)

In [ ]:
class BIEM:
    def __init__(self, cfg): self.cfg = cfg
    def assess(self, sim, Dsafe):
        c = sim.cfg; cap = c['cap_kwh']; healthy = sim.soh >= c['soh_min']
        ready = sim.ready_mask(); n_ready = int(ready.sum())
        hrs = np.maximum(c['soc_req'] - sim.soc, 0)*cap*sim.soh/(0.7*cap*c['eta'])           # time to ready at nominal 0.7C
        n_soon = min(int((~ready & healthy & (hrs <= 1.0)).sum()), int(c['p_max']/(0.7*cap)))
        I_eff = n_ready + n_soon                                                              # Eq. (23)
        N_req = int(np.ceil(np.sum(Dsafe)))                                                   # Eq. (24)
        deficit = max(N_req - I_eff, 0); surplus = max(I_eff - N_req, 0)                      # Eqs. (25)-(26)
        decision = 'charge_more' if deficit > 0 else ('defer_charging' if surplus > 0.5*N_req else 'maintain')   # Eq. (27)
        return dict(n_ready=n_ready, n_soon=n_soon, I_eff=I_eff, N_req=N_req, deficit=deficit, surplus=surplus,
                    decision=decision, pressure=min(deficit/(N_req + 1), 1.0))

class PBAM:
    """Predictive battery allocation: feasible set (28), one battery per EV (29)-(30), score (31), Hungarian optimum (32)."""
    def __init__(self, bhaam, w=(0.30, 0.20, 0.20, 0.15, 0.15)):
        assert abs(sum(w) - 1) < 1e-9; self.bh, self.w = bhaam, w
    def allocate(self, sim, evs, t, ctx, Dsafe_total):
        feas = self.bh.feasible(sim); cand = np.where(feas)[0]
        if len(cand) == 0: return [-1]*len(evs)
        soc_n, soh_n, d_n, cy_n = [a[cand] for a in self.bh.terms(sim)]
        # reserve term: prefer to give batteries of the type with surplus relative to forecast demand share
        ready_k = np.array([(feas & (sim.typ == k)).sum() for k in range(3)])
        need_k = np.array([ETYPE_P[COMPAT[:, k]].sum() for k in range(3)])*Dsafe_total   # demand that battery type k can serve
        scarcity = need_k/(ready_k + 1.0); res_n = 1 - scarcity[sim.typ[cand]]/(scarcity.max() + 1e-9)
        w = self.w; s = w[0]*soc_n + w[1]*soh_n - w[2]*d_n - w[3]*cy_n + w[4]*res_n           # Eq. (31)
        S = np.full((len(evs), len(cand)), -1e6)
        for e, (et, _) in enumerate(evs):
            comp = COMPAT[et, sim.typ[cand]]; S[e, comp] = s[comp]                              # Eq. (28) compatibility
        rows, cols = linear_sum_assignment(-S)                                                  # Eq. (32)
        out = [-1]*len(evs)
        for r_, c_ in zip(rows, cols):
            if S[r_, c_] > -1e5: out[r_] = int(cand[c_])
        return out

Calibrate the degradation estimator on a 5-day random-policy run.

In [ ]:
def make_scenario(seed, days=None):
    days = days or CFG['sim_days']; df = make_series(days + 2, seed=seed)       # +1 day warm-up for look-back, +1 for horizon
    return df

def calibrate_degradation():
    df = make_series(5, seed=999); rng = np.random.default_rng(5)
    sim = Station(CFG, init_fleet(np.random.default_rng(6), EVIOT), df, rng, calib=True); pol = RandomPolicy(1)
    for t in range(0, len(df) - 1): sim.step(t, pol, {})
    est = DegEstimator().fit(sim.log); print(f'degradation estimator fitted on {len(sim.log)} charging records')
    return est
DEG = calibrate_degradation()
print('estimated loss per unit-SOC  fresh pack 0.7C:', DEG.rate(1.0, 30, 0.7), ' weak pack 0.7C:', DEG.rate(0.75, 30, 0.7), ' weak pack 1.0C hot:', DEG.rate(0.75, 42, 1.0))

## 8. AMOO — Adaptive Multi-Objective Optimisation, equations (37)–(44), (52)–(53)
At every hour AMOO evaluates a grid of charging actions (power-budget fraction × per-battery C-rate) by a forecast-driven H-hour roll-out
and scores them with the four objectives of Eqs. (37)–(40): cost J1, waiting J2, degradation J3, inventory reliability J4.
Objectives are min-max normalised across candidates and combined with **state-dependent weights** (Eqs. 41-44):
high demand -> waiting/inventory weight up; low renewable or high price -> cost weight up; degraded fleet -> degradation weight up.
Feedback (Eqs. 52-53): EWMA forecast-error correction, adaptive safety coefficient κ and weight-bias updates from observed shortages.
J1 includes a shadow value of stored energy (mean tariff) so that charging is shifted to cheap / renewable hours instead of being postponed forever.

In [ ]:
class AMOO:
    def __init__(self, cfg, deg, kappa0=1.0, use_feedback=True, use_inv=True):
        self.cfg, self.deg, self.kappa0, self.use_feedback, self.use_inv = cfg, deg, kappa0, use_feedback, use_inv
        self.cands = [(f, c) for f in (0.2, 0.4, 0.6, 0.8, 1.0) for c in (0.4, 0.7, 1.0)]
        self.reset()
    def reset(self):
        self.b = np.zeros(4); self.kappa = self.kappa0; self.bias_d = 0.0; self.bias_r = 0.0
        self.err_scale = 1.0; self.short_ewma = 0.02; self.abs_e = 1.0; self.sg_e = 1.0
        self.w_hist = []; self.rec_cands = None

    def weights(self, s):                                                      # Eq. (44), satisfies (42)-(43)
        z = np.array([1.6*(1 - s['ren']) + 1.4*s['price'] - 1.0*s['dem'],      # cost
                      2.0*s['dem'] + 1.2*s['inv'],                             # waiting
                      2.0*s['health'] - 0.5*s['dem'],                          # degradation
                      1.8*s['inv'] + 1.0*s['dem']]) + self.b                   # inventory reliability
        e = np.exp(z - z.max()); p = e/e.sum(); eps = 0.05
        return (1 - 4*eps)*p + eps

    def rollout(self, sim, f, crate, Dsafe, Rh, price, amb, p_ref):
        cfg = self.cfg; cap = cfg['cap_kwh']; eta = cfg['eta']; Hn = len(Dsafe)
        ok = sim.soh >= cfg['soh_min']; soh_m = float(sim.soh[ok].mean()) if ok.any() else float(sim.soh.mean())
        need = np.sort(np.maximum(cfg['soc_target'] - sim.soc[ok], 0)*cap*sim.soh[ok])
        thr = (cfg['soc_target'] - cfg['soc_req'])*cap*soh_m; e_ret = 0.80*cap*soh_m
        pcap = crate*cfg['c_max']*cap*0.85
        drate = float(self.deg.rate(soh_m, amb + 18*crate**2, crate, 0.6))
        cost = deg = short = imb = deliv_tot = 0.0
        for h in range(Hn):
            nr = int((need <= thr).sum()); served = min(Dsafe[h], nr); short += max(Dsafe[h] - nr, 0)
            k = int(round(served))
            if k > 0: need = np.sort(np.concatenate([need[k:], np.full(k, e_ret)]))
            pi = np.minimum(pcap, need/eta); cum = np.cumsum(pi); p = np.clip(f*cfg['p_max'] - (cum - pi), 0, pi)
            need = need - p*eta; deliv = float(p.sum()); deliv_tot += deliv
            grid = max(deliv + cfg['aux_kw'] - Rh[h], 0.0); cost += price[h]*grid
            deg += deliv*eta/(cap*soh_m)*drate
            if self.use_inv:
                nr2 = int((need <= thr).sum()); imb += max(Dsafe[min(h + 1, Hn - 1)] - nr2, 0)
        cost -= p_ref*eta*deliv_tot                                           # shadow value of stored energy
        if self.use_inv:
            nr_end = int((need <= thr).sum()); imb += max(2*float(np.mean(Dsafe)) - nr_end, 0)
        return np.array([cost, short*1.0, deg, short*3.0 + imb])             # J1..J4  (wait ~ EV-hours; J4 = penalty + imbalance)

    def choose(self, sim, t, ctx, mu, sig, Rh, price, amb, inv):
        cfg = self.cfg; Dsafe = mu + self.kappa*sig                           # Eq. (11)
        p_ref = float(ctx['df'].price.mean())
        s = dict(dem=np.clip(np.mean(Dsafe)/12, 0, 1), ren=np.clip(np.mean(Rh)/(0.6*cfg['pv_kwp']), 0, 1),
                 price=np.clip((price[0] - 0.08)/(0.26 - 0.08), 0, 1), health=np.clip((1 - sim.soh.mean())/0.25, 0, 1),
                 inv=inv['pressure'] if self.use_inv else 0.0)
        w = self.weights(s)
        J = np.array([self.rollout(sim, f, c, Dsafe, Rh, price, amb, p_ref) for f, c in self.cands])
        Jn = np.stack([minmax01(J[:, k]) for k in range(4)], 1); tot = Jn @ w   # Eq. (41)
        k = int(np.argmin(tot)); self.w_hist.append(w)
        if ctx.get('record_t') == t: self.rec_cands = dict(cands=self.cands, J=J, Jn=Jn, w=w, tot=tot, k=k)
        return self.cands[k]

    def feedback(self, sim, t, ctx, obs, mu0, sg0, R0):
        if not self.use_feedback: return
        a = 0.2; e_d = obs['n_new'] - mu0; e_r = obs['pv'] - R0                 # Eqs. (52)-(53)
        self.bias_d = (1 - a)*self.bias_d + a*e_d; self.bias_r = (1 - a)*self.bias_r + a*e_r
        if sg0 > 1e-6:
            self.abs_e = (1 - a)*self.abs_e + a*abs(e_d); self.sg_e = (1 - a)*self.sg_e + a*sg0
            self.err_scale = float(np.clip(self.abs_e/(self.sg_e + 1e-6), 0.7, 2.5))
        self.short_ewma = 0.9*self.short_ewma + 0.1*(obs['first_fail']/max(obs['n_new'], 1))
        if self.short_ewma > 0.03:
            self.b[1] += 0.05; self.b[3] += 0.05; self.b[0] -= 0.03; self.kappa += 0.05
        elif self.short_ewma < 0.005:
            self.b[0] += 0.01; self.b[1] -= 0.01; self.b[3] -= 0.01; self.kappa -= 0.01
        self.b = np.clip(self.b, -0.8, 0.8); self.kappa = float(np.clip(self.kappa, 0.0, 3.0))

## 9. Policies: HAPO-BSS (with ablation switches) and comparison baselines
* **Static scheduling**: always charge at full power; SOC-greedy allocation.
* **SOC-only optimisation**: price-aware charging, SOC-greedy allocation.
* **MILP-surrogate** (stands in for [28]): deterministic day-ahead LP on the historical mean demand profile.
* **LSTM-RHO-surrogate** (stands in for [25]): rolling-horizon LP driven by the LSTM (or ridge) forecast, SOC-greedy allocation.
* **AGWO-surrogate** (stands in for [26]): adaptive Grey-Wolf search tuning a parametric charging rule on a separate training episode.
* GSO-PPO [23] is not re-implemented (needs an RL training environment); `StaticPolicy`-style hooks make adding one straightforward.

In [ ]:
class HAPO:
    def __init__(self, deg, name='HAPO-BSS', use_tdpm=True, use_rgemm=True, use_bhaam=True, use_biem=True, use_feedback=True):
        self.name = name; self.use_tdpm, self.use_rgemm, self.use_bhaam, self.use_biem, self.use_fb = use_tdpm, use_rgemm, use_bhaam, use_biem, use_feedback
        self.bhaam = BHAAM(deg); self.pbam = PBAM(self.bhaam); self.biem = BIEM(CFG)
        self.amoo = AMOO(CFG, deg, use_feedback=use_feedback, use_inv=use_biem)
    def reset(self, sim, ctx): self.amoo.reset(); self._pt = -1
    def _prep(self, sim, t, ctx):
        if self._pt == t: return
        Hh = CFG['H']; a = self.amoo
        if self.use_tdpm: mu0, sg0 = ctx['mu_d'][t].copy(), ctx['sg_d'][t].copy()
        else: mu0, sg0 = np.full(Hh, ctx['df'].demand.iat[t-1], float), np.zeros(Hh)          # persistence forecast
        R0 = ctx['mu_r'][t].copy() if self.use_rgemm else np.zeros(Hh)
        self.mu0, self.sg0, self.R0 = mu0, sg0, R0
        self.mu = np.maximum(mu0 + a.bias_d, 0); self.sig = sg0*a.err_scale
        self.Rh = np.maximum(R0 + (a.bias_r if self.use_rgemm else 0.0), 0); self.price = ctx['df'].price.values[t:t+Hh]; self._pt = t
    def allocate(self, sim, evs, t, ctx):
        if not self.use_bhaam: return greedy_soc_alloc(sim, evs)
        self._prep(sim, t, ctx); Dsafe_total = float(np.sum(self.mu + self.amoo.kappa*self.sig))
        return self.pbam.allocate(sim, evs, t, ctx, Dsafe_total)
    def charge_action(self, sim, t, ctx):
        self._prep(sim, t, ctx); Dsafe = self.mu + self.amoo.kappa*self.sig
        inv = self.biem.assess(sim, Dsafe); self.inv = inv
        f, c = self.amoo.choose(sim, t, ctx, self.mu, self.sig, self.Rh, self.price, float(ctx['df'].temp.iat[t]), inv)
        return f, c, ('health' if self.use_bhaam else 'soc')
    def after_step(self, sim, t, ctx, obs):
        self.amoo.feedback(sim, t, ctx, obs, self.mu0[0], self.sg0[0], self.R0[0])

class StaticPolicy:
    name = 'Static scheduling'
    def reset(self, sim, ctx): pass
    def allocate(self, sim, evs, t, ctx): return greedy_soc_alloc(sim, evs)
    def charge_action(self, sim, t, ctx): return 1.0, 1.0, 'soc'
    def after_step(self, sim, t, ctx, obs): pass

class SOCOnlyPolicy(StaticPolicy):
    name = 'SOC-only optimization'
    def charge_action(self, sim, t, ctx):
        p = ctx['df'].price.iat[t]; return (1.0 if p <= ctx['price_med'] else 0.35), 1.0, 'soc'

def lp_plan(S0, d, R, price, p_ref, floor, S_max, cfg):
    """min sum price*g - p_ref*sum x  s.t. energy-pool dynamics, charger limit, g >= x/eta + aux - R."""
    Hn = len(d); eta = cfg['eta']
    c = np.concatenate([-p_ref*np.ones(Hn), np.asarray(price, float)])
    cumd = np.cumsum(d); A, b = [], []
    for h in range(Hn):
        row = np.zeros(2*Hn); row[:h+1] = -1; A.append(row); b.append(S0 - cumd[h] - floor[h])            # S_h >= floor_h
        row = np.zeros(2*Hn); row[:h+1] = 1;  A.append(row); b.append(S_max - S0 + cumd[h])                # S_h <= S_max
        row = np.zeros(2*Hn); row[h] = 1/eta; row[Hn + h] = -1; A.append(row); b.append(R[h] - cfg['aux_kw'])
    bounds = [(0, eta*cfg['p_max'])]*Hn + [(0, None)]*Hn
    res = linprog(c, A_ub=np.array(A), b_ub=np.array(b), bounds=bounds, method='highs')
    return res.x[:Hn] if res.success else None

def pool_energy(sim):
    c = sim.cfg; cap = c['cap_kwh']; S0 = float(np.sum(np.maximum(sim.soc - 0.17, 0)*cap*sim.soh))
    S_max = float(np.sum((c['soc_target'] - 0.17)*cap*sim.soh)); e_sw = (c['soc_target'] - 0.17)*cap*float(sim.soh.mean())
    return S0, S_max, e_sw

class MILPSurrogate(StaticPolicy):
    name = 'MILP (deterministic LP surrogate)'
    def reset(self, sim, ctx): self.plan = None
    def charge_action(self, sim, t, ctx):
        h = int(ctx['df'].hour.iat[t])
        if self.plan is None or h == 0 or self.plan_t0 is None:
            S0, S_max, e_sw = pool_energy(sim); prof = ctx['hour_profile']; d = prof[(np.arange(24) + h) % 24]*e_sw
            R = ctx['pv_profile'][(np.arange(24) + h) % 24]; price = ctx['df'].price.values[t:t+24]
            if len(price) < 24: price = np.pad(price, (0, 24 - len(price)), mode='edge')
            floor = np.array([1.2*d[k:k+3].sum() for k in range(24)])
            x = lp_plan(S0, d, R, price, float(ctx['df'].price.mean()), floor, S_max, CFG)
            self.plan = x if x is not None else np.full(24, CFG['eta']*CFG['p_max']); self.plan_t0 = t
        k = min(t - self.plan_t0, 23); return float(np.clip(self.plan[k]/CFG['eta']/CFG['p_max'], 0.05, 1.0)), 1.0, 'soc'
    plan_t0 = None

class RHOSurrogate(StaticPolicy):
    name = 'LSTM-RHO (rolling-horizon LP surrogate)'
    def charge_action(self, sim, t, ctx):
        Hh = CFG['H']; S0, S_max, e_sw = pool_energy(sim); d = np.maximum(ctx['mu_b'][t], 0)*e_sw
        R = ctx['mu_r'][t]; price = ctx['df'].price.values[t:t+Hh]
        floor = np.array([1.2*d[k:k+2].sum() for k in range(Hh)])
        x = lp_plan(S0, d, R, price, float(ctx['df'].price.mean()), floor, S_max, CFG)
        f = 1.0 if x is None else float(np.clip(x[0]/CFG['eta']/CFG['p_max'], 0.05, 1.0)); return f, 1.0, 'soc'

class ParamPolicy(StaticPolicy):
    name = 'AGWO (adaptive grey-wolf surrogate)'
    def __init__(self, theta=None): self.theta = theta if theta is not None else np.array([0.5, 0.3, 0.3, 0.5])
    def charge_action(self, sim, t, ctx):
        a0, a1, a2, a3 = self.theta; d = ctx['df']
        price_n = np.clip((d.price.iat[t] - 0.08)/(0.18), 0, 1); pv_n = d.pv.iat[t]/CFG['pv_kwp']
        press = np.clip((24 - sim.ready_mask().sum())/24, 0, 1)
        return float(np.clip(a0 + a1*(1 - price_n) + a2*pv_n + a3*press, 0.1, 1.0)), 1.0, 'soc'

def make_ctx(df, fc=None):
    ctx = dict(df=df, price_med=float(df.price.median()))
    prof = raw.iloc[:n_tr].groupby('hour').demand.mean().values; ctx['hour_profile'] = prof
    ctx['pv_profile'] = raw.iloc[:n_tr].groupby('hour').pv.mean().values
    if fc is None: fc = forecast_series(df)
    ctx.update(fc); return ctx

def run_episode(policy, df, ctx, seed, t0=None, t1=None):
    rng = np.random.default_rng(seed); fleet = init_fleet(np.random.default_rng(seed + 1000), EVIOT)
    sim = Station(CFG, fleet, df, rng); policy.reset(sim, ctx)
    t0 = L if t0 is None else t0; t1 = (len(df) - H) if t1 is None else t1
    for t in range(t0, t1): sim.step(t, policy, ctx)
    return sim.summary(), sim

def grey_wolf(fitness, lb, ub, n_wolves=8, iters=10, seed=0):
    """Grey Wolf Optimiser with a non-linear (adaptive) decay of the control parameter a."""
    r = np.random.default_rng(seed); lb, ub = np.array(lb), np.array(ub); X = r.uniform(lb, ub, (n_wolves, len(lb)))
    fit = np.array([fitness(x) for x in X])
    for it in range(iters):
        order = np.argsort(fit); alpha, beta, delta = X[order[0]].copy(), X[order[1]].copy(), X[order[2]].copy()
        a = 2*(1 - (it/iters)**2)
        for i in range(n_wolves):
            new = np.zeros(len(lb))
            for leader in (alpha, beta, delta):
                A = 2*a*r.random(len(lb)) - a; C = 2*r.random(len(lb)); new += leader - A*np.abs(C*leader - X[i])
            X[i] = np.clip(new/3, lb, ub); fit[i] = fitness(X[i])
    k = int(np.argmin(fit)); return X[k], fit[k]

def tune_agwo():
    df = make_series(7 + 2, seed=555); ctx = make_ctx(df, fc={k: np.zeros((len(df), H)) for k in ['mu_d', 'sg_d', 'mu_r', 'sg_r', 'mu_b']})
    def fitness(theta):
        s, _ = run_episode(ParamPolicy(theta), df, ctx, seed=3); return s['grid_cost'] + 20*s['lost_evs'] + 0.002*s['deg_cost']
    th, fv = grey_wolf(fitness, [0, -1, -1, 0], [1, 1, 1, 1]); print('AGWO tuned theta:', th.round(2), 'fitness', round(fv, 1)); return th

## 10. Experiments
Three independent stochastic scenarios (different weather / demand / fleet / arrival realisations). Forecasters are trained once on the main series and applied
online (history-only windows) in each scenario.

In [ ]:
SCEN = []
for k in range(CFG['n_scen']):
    df = make_scenario(seed=100 + k); SCEN.append((df, make_ctx(df), 200 + k))
AGWO_THETA = tune_agwo()
METRICS = ['total_cost', 'grid_cost', 'deg_cost', 'deg_loss_pct', 'lost_evs', 'avg_wait_min', 'BIAR', 'BSR', 'SFR', 'BASR',
           'RESSR', 'GEDR', 'BHRR', 'CCU', 'OCT_ms']

def evaluate(make_policy, scen=SCEN, keep_sim=False):
    res, sims = [], []
    for df, ctx, seed in scen:
        c = dict(ctx); c['record_t'] = L + 24 + 18                                   # record AMOO candidates on day 2, 18:00
        s, sim = run_episode(make_policy(), df, c, seed); res.append(s); sims.append(sim)
    d = pd.DataFrame(res)[METRICS]
    return (d, sims) if keep_sim else d

POLICIES = {
    'Static scheduling':          lambda: StaticPolicy(),
    'SOC-only optimization':      lambda: SOCOnlyPolicy(),
    'MILP (LP surrogate)':        lambda: MILPSurrogate(),
    'LSTM-RHO (surrogate)':       lambda: RHOSurrogate(),
    'AGWO (surrogate)':           lambda: ParamPolicy(AGWO_THETA),
    'HAPO-BSS (proposed)':        lambda: HAPO(DEG),
}
RESULTS, SIMS = {}, {}
for name, mk in POLICIES.items():
    t0 = time.time(); RESULTS[name], SIMS[name] = evaluate(mk, keep_sim=True); print(f'{name:28s} done in {time.time()-t0:5.1f}s')

def fmt(res, cols):
    return pd.DataFrame({n: [f'{d[c].mean():.2f} ± {d[c].std():.2f}' for c in cols] for n, d in res.items()}, index=cols).T
print('\n=== Table A: operating cost, degradation, waiting (mean ± std over scenarios) ===')
print(fmt(RESULTS, ['total_cost', 'grid_cost', 'deg_cost', 'deg_loss_pct', 'lost_evs', 'avg_wait_min']).to_string())
print('\n=== Table B (analogue of paper Table 2): reliability and sustainability ===')
print(fmt(RESULTS, ['BIAR', 'BSR', 'SFR', 'BASR', 'RESSR', 'GEDR']).to_string())
print('\n=== Table C: battery health, charger use, decision time ===')
print(fmt(RESULTS, ['BHRR', 'CCU', 'OCT_ms']).to_string())

In [ ]:
names = list(RESULTS); cols = ['#999999', '#bbbb55', '#55aa88', '#5588cc', '#aa77cc', '#d9534f']
def bar(ax, metric, title, scale=1.0):
    m = [RESULTS[n][metric].mean()*scale for n in names]; s = [RESULTS[n][metric].std()*scale for n in names]
    ax.bar(range(len(names)), m, yerr=s, color=cols, capsize=3); ax.set_xticks(range(len(names)))
    ax.set_xticklabels([n.split(' (')[0] for n in names], rotation=35, ha='right', fontsize=8); ax.set_title(title, fontsize=10)
fig, ax = plt.subplots(2, 4, figsize=(17, 7.5))
for a, (m, t) in zip(ax.ravel(), [('total_cost', 'Total cost ($)'), ('grid_cost', 'Grid energy cost ($)'), ('deg_loss_pct', 'Mean SOH loss (%-pts)'),
                                   ('BSR', 'Battery shortage rate (%)'), ('SFR', 'Immediate service rate (%)'), ('RESSR', 'Renewable self-sufficiency (%)'),
                                   ('GEDR', 'Grid dependency (%)'), ('avg_wait_min', 'Avg wait (min)')]): bar(a, m, t)
show('fig_policy_comparison')

### Relative improvement of HAPO-BSS (Fig. 10–16 analogue)

In [ ]:
def rel(base, m):
    b = RESULTS[base][m].mean(); h = RESULTS['HAPO-BSS (proposed)'][m].mean()
    return round(100*(1 - h/b), 1) if abs(b) > 1e-6 else np.nan                  # NaN: baseline value is zero
mets = ['grid_cost', 'deg_loss_pct', 'total_cost', 'avg_wait_min', 'lost_evs']
imp = pd.DataFrame({'metric': mets, **{f'HAPO vs {b} (%)': [rel(b, m) for m in mets] for b in ['Static scheduling', 'SOC-only optimization', 'MILP (LP surrogate)']}})
print('Positive = HAPO-BSS lower (better); negative = HAPO-BSS higher; NaN = baseline is 0.'); print(imp.to_string(index=False))

## 11. Inside HAPO-BSS: operating profile, adaptive weights, Pareto trade-off

In [ ]:
sim_h = SIMS['HAPO-BSS (proposed)'][0]; sim_s = SIMS['SOC-only optimization'][0]
rh = pd.DataFrame(sim_h.rec); rs = pd.DataFrame(sim_s.rec); span = slice(24, 24*4)
fig, ax = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
for r_, lab, col in [(rh, 'HAPO-BSS', 'r'), (rs, 'SOC-only', 'gray')]: ax[0].plot(r_.E_in.values[span], col, label=lab)
ax[0].plot(rh.pv.values[span], 'g--', label='PV (kW)'); ax[0].set_ylabel('charging energy / PV (kWh)'); ax[0].legend(ncol=3)
ax[1].plot(rh.price.values[span], 'k'); ax[1].set_ylabel('tariff ($/kWh)')
ax[2].plot(rh.n_ready.values[span], 'r', label='ready batteries HAPO'); ax[2].plot(rs.n_ready.values[span], 'gray', label='ready batteries SOC-only')
ax[2].plot(rh.n_evs.values[span], 'b.', label='EV demand'); ax[2].legend(ncol=3); ax[2].set_xlabel('hour'); show('fig_profile')

pol_h = HAPO(DEG); _ = run_episode(pol_h, SCEN[0][0], dict(SCEN[0][1], record_t=L + 24 + 18), SCEN[0][2])
W = np.array(pol_h.amoo.w_hist); fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
for k, n in enumerate(['cost', 'waiting', 'degradation', 'inventory']): ax[0].plot(W[:96, k], label=n)
ax[0].set_title('AMOO adaptive weights (first 4 days)'); ax[0].set_xlabel('hour'); ax[0].legend(ncol=4, fontsize=8)
rc = pol_h.amoo.rec_cands
if rc is not None:
    J = rc['Jn']; ax[1].scatter(J[:, 0], J[:, 2], c=J[:, 3], cmap='viridis', s=60)
    ax[1].scatter(J[rc['k'], 0], J[rc['k'], 2], marker='*', s=300, c='r', label='chosen')
    for (f_, c_), x, y in zip(rc['cands'], J[:, 0], J[:, 2]): ax[1].annotate(f'{f_:.1f}/{c_:.1f}', (x, y), fontsize=6)
    ax[1].set_xlabel('normalised cost J1'); ax[1].set_ylabel('normalised degradation J3'); ax[1].set_title('Candidate actions at 18:00 (f/C-rate), colour = J4'); ax[1].legend()
show('fig_amoo')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
for n, col in [('SOC-only optimization', 'gray'), ('HAPO-BSS (proposed)', 'r')]:
    s = SIMS[n][0]; ax[0].hist(s.soh*100, bins=15, alpha=.55, color=col, label=n)
    day = pd.DataFrame(s.rec).assign(day=lambda d: (d.t - L)//24).groupby('day')
    ax[1].plot(day.cost.sum().values, color=col, label=n + ' (daily grid cost)')
ax[0].set_title('Final fleet SOH distribution (scenario 1)'); ax[0].set_xlabel('SOH (%)'); ax[0].legend(fontsize=8)
ax[1].set_title('Daily grid cost over the evaluation window'); ax[1].set_xlabel('day'); ax[1].legend(fontsize=8); show('fig_soh_daily')

## 12. Ablation study (analogue of Table 4)

In [ ]:
ABL = {
    'Full HAPO-BSS':        lambda: HAPO(DEG),
    'Without TDPM':         lambda: HAPO(DEG, use_tdpm=False),
    'Without RGEMM':        lambda: HAPO(DEG, use_rgemm=False),
    'Without BHAAM + PBAM': lambda: HAPO(DEG, use_bhaam=False),
    'Without BIEM':         lambda: HAPO(DEG, use_biem=False),
    'Without AMOO feedback':lambda: HAPO(DEG, use_feedback=False),
}
ABL_RES = {'Full HAPO-BSS': RESULTS['HAPO-BSS (proposed)']}
for n, mk in ABL.items():
    if n == 'Full HAPO-BSS': continue
    t0 = time.time(); ABL_RES[n] = evaluate(mk); print(f'{n:24s} {time.time()-t0:5.1f}s')
print(fmt(ABL_RES, ['total_cost', 'grid_cost', 'deg_loss_pct', 'BASR', 'BSR', 'BHRR', 'CCU', 'RESSR', 'OCT_ms']).to_string())

## 13. Constraint verification — equations (45)–(51)

In [ ]:
rows = []
for n, sims in SIMS.items():
    v = defaultdict(int)
    for s in sims:
        for k_, x in s.viol.items(): v[k_] += x
        r_ = pd.DataFrame(s.rec)
        v['soh_below_min'] += int((s.soh < CFG['soh_min']).sum())
    rows.append(dict(policy=n, charger_capacity=v['charger_capacity'], soc_range=v['soc_range'], double_allocation=v['double_allocation'],
                     renewable=v['renewable'], grid_power=v['grid_power'], soh_below_min_batteries=v['soh_below_min']))
print(pd.DataFrame(rows).to_string(index=False))
print('\nAll hard constraints (battery availability, SOC range, charger capacity, renewable <= generation, grid power limit) hold when the counts above are zero.')
print('Inventory safety (Eq. 51) is a soft constraint enforced through the J4 objective; see BIAR/BSR above.')

## 14. Summary tables for the paper
Copy `TABLE_FORECAST`, the Table A/B/C prints and the ablation table into the manuscript **as produced by your run**. The values are
simulation-dependent; change `CFG`, the scenario seeds or plug in real station logs (replace `make_series`) to study sensitivity.

**Extending to real data:** replace `make_series` by a loader that returns a DataFrame with columns
`demand, pv, price, temp, cloud, hour, dow` (hourly), keep `init_fleet` reading EVIoT SOC/SOH/temperature/cycles, and re-run.